> **Status: PREPARED — NOT YET EXECUTED**  
> Skeleton notebook ready. Requires `03_results/metrics_table.csv` from A2_01.  
> Written interpretation sections are scaffolded — complete them with your analysis.

# MaCAD S.3 — Assignment 2
## A2-02: Spatial Interpretation of Graph Metrics — Double-L Residential Building

**Assignment objective:** Translate each graph metric into an architectural spatial argument.
For each metric, explain what it measures topologically, how it behaves in the Double-L
building, and what spatial design knowledge it reveals.

**Course workflow:** S03 — Spatial Intelligence  
**Reference notebooks (read-only):**

| Notebook | Interpretation topics |
|----------|----------------------|
| `S03-07 Spatial Intelligence Part 1.ipynb` | Betweenness / closeness as circulation efficiency |
| `S03-08 Spatial Intelligence Part 2.ipynb` | Community detection as spatial zones / apartments |

**Prerequisite:** `A2_01_DoubleL_Graph_Metrics.ipynb` must be run first.

---

| Input | Location |
|-------|----------|
| `metrics_table.csv` | `03_results/` |
| `graph_summary.csv` | `03_results/` |
| `nodes.csv` | `01_input_graph/` or A1 `04_graph_dataset/` |
| `edges.csv` | `01_input_graph/` or A1 `04_graph_dataset/` |

| Output | Location |
|--------|----------|
| Composite comparison figure | `04_visuals/06_metric_comparison.png` |
| Per-role metric breakdown | `04_visuals/07_metrics_by_role.png` |
| Interpretation text | `05_submission_text/interpretation.md` |

---
## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

matplotlib.rcParams["figure.dpi"] = 120

PROJECT_ROOT = "D:/GitHub/GML_Edu"

A1_ROOT    = os.path.join(PROJECT_ROOT, "assignments", "assignment_01_graph_generation")
A1_DATASET = os.path.join(A1_ROOT, "04_graph_dataset")

A2_ROOT      = os.path.join(PROJECT_ROOT, "assignments", "assignment_02_graph_analysis")
INPUT_DIR    = os.path.join(A2_ROOT, "01_input_graph")
RESULTS_DIR  = os.path.join(A2_ROOT, "03_results")
VISUALS_DIR  = os.path.join(A2_ROOT, "04_visuals")
SUBMISSION   = os.path.join(A2_ROOT, "05_submission_text")

os.makedirs(VISUALS_DIR, exist_ok=True)
os.makedirs(SUBMISSION,  exist_ok=True)

METRICS_CSV = os.path.join(RESULTS_DIR, "metrics_table.csv")
SUMMARY_CSV = os.path.join(RESULTS_DIR, "graph_summary.csv")

NODES_CSV = (
    os.path.join(INPUT_DIR,  "nodes.csv") if os.path.exists(os.path.join(INPUT_DIR, "nodes.csv"))
    else os.path.join(A1_DATASET, "nodes.csv")
)
EDGES_CSV = (
    os.path.join(INPUT_DIR,  "edges.csv") if os.path.exists(os.path.join(INPUT_DIR, "edges.csv"))
    else os.path.join(A1_DATASET, "edges.csv")
)

for label, path in [("metrics_table.csv", METRICS_CSV),
                    ("graph_summary.csv",  SUMMARY_CSV),
                    ("nodes.csv",          NODES_CSV),
                    ("edges.csv",          EDGES_CSV)]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {label:22s}: [{status}]")

if not os.path.exists(METRICS_CSV):
    raise FileNotFoundError(
        f"metrics_table.csv not found: {METRICS_CSV}\n"
        "Run A2_01_DoubleL_Graph_Metrics.ipynb first."
    )

In [ ]:
metrics  = pd.read_csv(METRICS_CSV)
summary  = pd.read_csv(SUMMARY_CSV)
nodes    = pd.read_csv(NODES_CSV)
edges    = pd.read_csv(EDGES_CSV)

print(f"metrics_table : {len(metrics)} rows, columns: {list(metrics.columns)}")
print(f"graph_summary :")
print(summary.T.to_string())

# Rebuild NetworkX for plan layouts
G = nx.Graph()
for _, row in nodes.iterrows():
    G.add_node(int(row["node_id"]), **row.to_dict())
for _, row in edges.iterrows():
    G.add_edge(int(row["src_id"]), int(row["dst_id"]))

pos_xy = {int(row["node_id"]): (float(row["X"]), float(row["Y"]))
          for _, row in nodes.iterrows()}

print(f"\nGraph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

---
## 2. Composite Metric Comparison

A 2×3 subplot panel showing all five metrics at once in plan view.
This gives an overview of how each metric distributes across the building.

In [ ]:
m_df = metrics.set_index("node_id")

METRIC_COLS = [
    ("degree_cent", "Degree Centrality"),
    ("betweenness", "Betweenness Centrality"),
    ("closeness",   "Closeness Centrality"),
    ("clustering",  "Clustering Coefficient"),
    ("community",   "Community"),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes_flat = axes.flatten()

for idx, (col, title) in enumerate(METRIC_COLS):
    ax   = axes_flat[idx]
    vals = [m_df.loc[n, col] if n in m_df.index else 0.0 for n in G.nodes()]
    cmap = plt.cm.tab20 if col == "community" else plt.cm.YlOrRd
    norm = Normalize(vmin=min(vals), vmax=max(max(vals), 1e-9))
    cols = [cmap(norm(v)) for v in vals]

    nx.draw_networkx_edges(G, pos_xy, ax=ax, alpha=0.12, edge_color="#999", width=0.3)
    nx.draw_networkx_nodes(G, pos_xy, ax=ax, node_color=cols, node_size=20, edgecolors="none")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, shrink=0.5)
    ax.set_title(title, fontsize=10)
    ax.axis("off")

# Hide the unused 6th panel
axes_flat[5].axis("off")

plt.suptitle("Double-L TB_01 — Graph Metric Overview", fontsize=13, y=1.01)
plt.tight_layout()
out = os.path.join(VISUALS_DIR, "06_metric_comparison.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

---
## 3. Metric Breakdown by Node Role

Box plots showing the distribution of each metric per node role
(room / corridor / core). This is the core spatial argument:
corridor and core nodes should rank higher on betweenness and closeness
if the graph correctly represents a hierarchical circulation network.

In [ ]:
metric_cols = ["degree_cent", "betweenness", "closeness", "clustering"]
roles       = metrics["node_role"].unique()

fig, axes = plt.subplots(1, len(metric_cols), figsize=(16, 5), sharey=False)

for ax, col in zip(axes, metric_cols):
    groups = [metrics[metrics["node_role"] == r][col].dropna().values for r in roles]
    bp     = ax.boxplot(groups, labels=roles, patch_artist=True,
                        medianprops=dict(color="black", linewidth=1.5))
    colors_box = ["#81C784", "#4FC3F7", "#E57373"][:len(roles)]
    for patch, c in zip(bp["boxes"], colors_box):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_title(col.replace("_", " ").title(), fontsize=10)
    ax.set_ylabel("Value")
    ax.tick_params(axis="x", labelsize=9)

plt.suptitle("Metric Distribution by Node Role — Double-L TB_01", fontsize=12)
plt.tight_layout()
out = os.path.join(VISUALS_DIR, "07_metrics_by_role.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

---
## 4. Per-Floor Metric Profile

Mean metric values per floor. A symmetric residential building should show
similar per-floor profiles. Deviations indicate edge effects (ground floor,
roof floor) or missing geometry (inter-floor stair links).

In [ ]:
floor_metrics = metrics.groupby("floor_id")[["degree_cent", "betweenness", "closeness", "clustering"]].mean()
print("Mean metric values per floor:")
print(floor_metrics.round(4).to_string())

floor_metrics.plot(kind="bar", figsize=(12, 5), colormap="tab10", edgecolor="white")
plt.title("Mean Metric Values per Floor — Double-L TB_01", fontsize=12)
plt.xlabel("Floor ID")
plt.ylabel("Mean value")
plt.xticks(rotation=0)
plt.legend(fontsize=9, loc="upper right")
plt.tight_layout()
out = os.path.join(VISUALS_DIR, "08_floor_metric_profile.png")
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

---
## 5. Spatial Interpretation — Degree Centrality

**Topological meaning:** Degree centrality counts the proportion of other nodes
directly adjacent to a given node.

**In this building:**  
*Complete this section with your analysis after running the metrics.*

Typical findings in a double-loaded corridor residential:
- Corridor nodes should have the highest degree (one shared face per apartment hall)
- Hall nodes typically rank second (connected to kitchen, bath, living, corridor)
- Bath and Bedroom nodes rank lowest (fewer shared faces)

**Spatial design reading:**  
High degree indicates a spatially promiscuous element — one that touches many others.
In a residential block this is the corridor acting as a distribution node.

In [ ]:
print("Degree centrality by space type:")
dc_by_type = metrics.groupby("space_type")["degree_cent"].agg(["mean", "max", "min", "count"])
print(dc_by_type.sort_values("mean", ascending=False).round(4).to_string())
print()
print("Degree centrality by node role:")
print(metrics.groupby("node_role")["degree_cent"].mean().round(4).to_string())

---
## 6. Spatial Interpretation — Betweenness Centrality

**Topological meaning:** The fraction of all shortest paths between any two
nodes in the graph that pass through a given node. High betweenness = bottleneck.

**In this building:**  
*Complete this section with your analysis after running the metrics.*

Expected:
- Corridor nodes dominate betweenness — they sit on most inter-apartment paths
- Core nodes (stairs/lifts) are critical bottlenecks for inter-floor routes
- Rooms on the "waist" of the Double-L may show elevated betweenness

**Spatial design reading:**  
Betweenness reveals where spatial demand concentrates. A corridor with very
high betweenness is a congestion risk; redundant paths would lower the score.

In [ ]:
print("Betweenness centrality by node role:")
print(metrics.groupby("node_role")["betweenness"].agg(["mean", "max"]).round(4).to_string())
print()
print("Top 10 betweenness nodes:")
top_bc = metrics.nlargest(10, "betweenness")[["node_id","node_role","space_type","floor_id","betweenness"]]
print(top_bc.to_string(index=False))

---
## 7. Spatial Interpretation — Closeness Centrality

**Topological meaning:** The reciprocal of the average shortest-path length
from a given node to all others. High closeness = spatially central.

**In this building:**  
*Complete this section with your analysis after running the metrics.*

Expected:
- Corridors near the building centroid score highest
- Rooms at the ends of wings score lowest
- In the Double-L, the "elbow" rooms/corridors connecting the two wings should be central

**Spatial design reading:**  
Closeness maps spatial "centrality" in a way that mirrors the architect's intuition
about which rooms are easiest to reach from anywhere. It is a graph proxy for
accessibility.

In [ ]:
print("Closeness centrality by node role:")
print(metrics.groupby("node_role")["closeness"].agg(["mean", "max"]).round(4).to_string())
print()
print("Top 10 closeness nodes:")
top_cc = metrics.nlargest(10, "closeness")[["node_id","node_role","space_type","floor_id","closeness"]]
print(top_cc.to_string(index=False))

---
## 8. Spatial Interpretation — Clustering Coefficient

**Topological meaning:** What proportion of a node's neighbours are also
neighbours of each other? High clustering = room embedded in a clique-like group.

**In this building:**  
*Complete this section with your analysis after running the metrics.*

Expected:
- Rooms within compact apartments (rectangular plan with shared walls) cluster tightly
- Corridor nodes have low clustering (neighbours = apartment halls, which don't touch each other)
- Clustering ≈ 0 for corridors is a graph signature of a hub/spoke or tree topology

**Spatial design reading:**  
Clustering reveals clique structure — rooms that form local closed groups.
In residential buildings this corresponds to room clusters within apartments.

In [ ]:
print("Clustering coefficient by node role:")
print(metrics.groupby("node_role")["clustering"].agg(["mean", "max", "min"]).round(4).to_string())
print()
print("Clustering coefficient by space type:")
print(metrics.groupby("space_type")["clustering"].mean().sort_values(ascending=False).round(4).to_string())

---
## 9. Spatial Interpretation — Community Detection

**Topological meaning:** Louvain algorithm partitions the graph into communities
that maximise within-community edge density relative to a random baseline.

**In this building:**  
*Complete this section with your analysis after running the metrics.*

Expected:
- Communities map onto apartment units (rooms that share many walls cluster together)
- Number of communities should be broadly consistent with floor × apartment count
- Corridor nodes may bridge multiple communities or form their own

**Spatial design reading:**  
Community detection is an unsupervised re-discovery of the architect's intended
spatial groupings. If communities align with apartments the graph correctly
encodes the building's ownership/use boundaries.

In [ ]:
n_comm     = metrics["community"].nunique()
comm_sizes = metrics.groupby("community").size().describe()

print(f"Communities detected  : {n_comm}")
print(f"Expected apartments   : {nodes['apartment_id'].nunique()} unique apartment IDs")
print()
print("Community size statistics:")
print(comm_sizes.round(1).to_string())
print()
print("Dominant space type per community:")
dom_type = metrics.groupby("community")["space_type"].agg(lambda x: x.value_counts().index[0])
print(dom_type.value_counts().to_string())

---
## 10. Correlation Matrix

Which metrics co-vary? High correlation between degree and betweenness
suggests the graph has a hub-spoke structure (consistent with corridor-dominated topology).

In [ ]:
metric_cols_num = ["degree_cent", "betweenness", "closeness", "clustering"]
corr_df = metrics[metric_cols_num].corr().round(3)
print("Pearson correlation matrix:")
print(corr_df.to_string())

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_df.values, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(metric_cols_num)))
ax.set_yticks(range(len(metric_cols_num)))
ax.set_xticklabels([c.replace("_", "\n") for c in metric_cols_num], fontsize=9)
ax.set_yticklabels([c.replace("_", " ") for c in metric_cols_num], fontsize=9)
for i in range(len(metric_cols_num)):
    for j in range(len(metric_cols_num)):
        ax.text(j, i, f"{corr_df.values[i,j]:.2f}", ha="center", va="center", fontsize=9)
plt.colorbar(im, ax=ax, fraction=0.04)
ax.set_title("Metric Correlation — Double-L TB_01", fontsize=11)
plt.tight_layout()
out = os.path.join(VISUALS_DIR, "09_metric_correlation.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out}")

---
## 11. Export Interpretation Template

Generates a markdown scaffold for your written submission. Complete the
`[YOUR ANALYSIS]` sections with architectural interpretation.

In [ ]:
n_nodes   = metrics["node_id"].nunique()
n_rooms   = len(metrics[metrics["node_role"] == "room"])
n_corr    = len(metrics[metrics["node_role"] == "corridor"])
n_core    = len(metrics[metrics["node_role"] == "core"])
density   = summary["density"].iloc[0] if "density" in summary.columns else "—"
diameter  = summary["diameter"].iloc[0] if "diameter" in summary.columns else "—"

interpretation_md = f"""\
# Assignment 2 — Spatial Interpretation
## Double-L Residential Building (TB_01)

**Course:** MaCAD S.3 — Buildings as Graphs  
**Student:** [YOUR NAME]  
**Date:** 2026-06-26

---

## Graph Overview

- Nodes: {n_nodes} (rooms: {n_rooms}, corridors: {n_corr}, cores: {n_core})
- Graph density: {density}
- Diameter: {diameter}

[YOUR ANALYSIS: describe what the density and diameter tell you about the
building's topological compactness. Is it a sparse or dense graph? What
does the diameter reveal about the maximum spatial distance?]

---

## Degree Centrality

**Definition:** Fraction of all other nodes that are directly adjacent.

[YOUR ANALYSIS: which space types have the highest degree? Why? How does
degree centrality reflect the double-loaded corridor spatial organisation?]

---

## Betweenness Centrality

**Definition:** Fraction of all shortest paths that pass through a node.

[YOUR ANALYSIS: which corridor or core has the highest betweenness? What
does this mean for building circulation? If that node were removed, what
would happen to spatial connectivity?]

---

## Closeness Centrality

**Definition:** Reciprocal of average shortest-path length to all others.

[YOUR ANALYSIS: which zones of the building are most "central" by this
metric? How does the Double-L footprint shape the closeness distribution?
Are the two wings equally central?]

---

## Clustering Coefficient

**Definition:** Proportion of a node's neighbours that are also neighbours
of each other.

[YOUR ANALYSIS: are corridor nodes high or low clustering? What does low
clustering in corridors reveal about the hub-spoke structure? Which room
types cluster most tightly?]

---

## Community Detection

**Definition:** Louvain partition that maximises within-community density.

[YOUR ANALYSIS: how many communities were detected? Do they correspond
to apartments, floors, or wings? Does the graph topology preserve the
architect's intended ownership boundaries?]

---

## Limitations

1. Graph built from shared-face adjacency (no door geometry in TB_01_doors.obj)
2. No vertical inter-floor links — stair/lift connections absent
3. Rooms separated by walls of non-zero thickness may not be connected

[YOUR ANALYSIS: how do these limitations affect the spatial reading?
Which metrics are most sensitive to the missing door/stair geometry?]
"""

interp_path = os.path.join(SUBMISSION, "interpretation.md")
with open(interp_path, "w", encoding="utf-8") as f:
    f.write(interpretation_md)
print(f"Interpretation template saved: {interp_path}")
print()
print("Fill in the [YOUR ANALYSIS] sections with your spatial argument.")

---
## 12. Verification and Final Checklist

In [ ]:
expected_outputs = [
    os.path.join(VISUALS_DIR, "06_metric_comparison.png"),
    os.path.join(VISUALS_DIR, "07_metrics_by_role.png"),
    os.path.join(VISUALS_DIR, "08_floor_metric_profile.png"),
    os.path.join(VISUALS_DIR, "09_metric_correlation.png"),
    os.path.join(SUBMISSION,  "interpretation.md"),
]
print("Output check:")
all_ok = True
for path in expected_outputs:
    status = "OK" if os.path.exists(path) else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  {os.path.basename(path):45s}: {status}")

print()
print("Assignment 2 checklist:")
checks = [
    "metrics_table.csv loaded and printed",
    "composite metric panel (06_metric_comparison.png) saved",
    "box plots by role (07_metrics_by_role.png) saved",
    "floor profile chart (08_floor_metric_profile.png) saved",
    "correlation matrix (09_metric_correlation.png) saved",
    "interpretation.md scaffold written to 05_submission_text/",
    "[YOUR ANALYSIS] sections filled in the markdown file",
]
for chk in checks:
    print(f"  [ ] {chk}")